Generate data trajectories $\boldsymbol{X}_{N+1}, \boldsymbol{U}_{N}$ on which the model rollout performance is evaluated
For each system generate trajectories based on...
- patterns generated through optimal control and full model knowledge

These patterns are to be stored in `admpe.data_management.TESTING_TRAJECTORIES_ROOT`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10.95})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)

figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 
import optax

from admpe.data_management import TESTING_TRAJECTORIES_ROOT

from admpe.excitation.excitation_experiment_setup import setup_environment

In [ ]:
# setup envs:
env, penalty_function, featurize, env_params, model_class = setup_environment(system_name="cart_pole")
env = eqx.tree_at(lambda s: s.process_noise_variance, env, 0.0)

#### generate testing patterns using an OCP formulation:

using parts of `DMPE.dmpe.utils.sets.reachable_set.py`

In [ ]:
from admpe.evaluation.test_trajectories.ocp_test_trajectories import (
    loss_function, get_candidate_batch_function
)

In [ ]:
seed = 5555

params = dict(
    n_steps=500,
    n_starts=4,
    deterministic_initial_state=False,
    n_opt_steps=10_000,
)

data_key = jax.random.key(seed)

lr = optax.schedules.exponential_decay(
    init_value=1e-1,
    transition_steps=int(params["n_opt_steps"] / 3),
    transition_begin=0,
    decay_rate=0.1,
    end_value=1e-4,
)
optimizer = optax.adam(lr)

In [ ]:
batch_candidate_function = get_candidate_batch_function(
    env, params, optimizer, featurize, loss_function, penalty_function, data_key
)

In [ ]:
candidate_ids = np.arange(0, 500, 1, dtype=np.uint32)
observations, chosen_actions, state_excess, action_excess, valid, proposed_actions, target_observation = batch_candidate_function(
    jnp.asarray(candidate_ids)
)

In [ ]:
observations.shape

In [ ]:
for obss in observations[:10]:
    plt.plot(obss)
    plt.show()

In [ ]:
jnp.mean(valid)

In [ ]:
plt.hist(action_excess, bins=100);

In [ ]:
plt.hist(state_excess, bins=100);

- store the results of this run
- then, build proper script including storing of results

In [ ]:
from uuid import uuid4
from admpe.data_management import (
    TESTING_TRAJECTORIES_ROOT,
    write_metadata,
    load_testing_sequences,
)
from admpe.evaluation.test_trajectories.ocp_test_trajectories import OCP_TESTING_SIGNALS_ROOT

In [ ]:
system_name = "cart_pole"
signal_name = "ocp"
state_tolerance = 1e-3
action_tolerance = 1e-3

root = OCP_TESTING_SIGNALS_ROOT

- visualization directly with the models is probably easier

In [ ]:
all_testing_sequences = load_testing_sequences(system_name)

In [ ]:
all_testing_sequences["ocp"].keys()